# Runtime

In [ ]:
import time

time_start = time.time()  ###
print(f'start_time: {time_start}')

start_time: 1757858236.278731


In [ ]:
time_end = time.time()  ###
print(f'runtime [**]: {time_end - time_start:.1f} [s]')  ###

start_time: 57.6 [s]


# PSNR

In [10]:
path_img1 = 'test/1305031102.175304.png'
path_img2 = 'test/1305031123.751723.png'

import torch
import torchvision.transforms as T
from PIL import Image

# Define a transform to convert PIL image to torch tensor and normalize to [0,1]
transform = T.Compose([
    T.ToTensor()
])

img1 = Image.open(path_img1).convert('RGB')
img2 = Image.open(path_img2).convert('RGB')

img1 = transform(img1)
img2 = transform(img2)

In [11]:
img1.shape, img2.shape

(torch.Size([3, 480, 640]), torch.Size([3, 480, 640]))

In [12]:
import torch.nn.functional as F
def psnr1(img1, img2):
    return 10 * torch.log10(1 / F.mse_loss(img1, img2)).item()

In [13]:
def psnr2(img1, img2):
    mse = ((img1 - img2) ** 2).view(img1.shape[0], -1).mean(1, keepdim=True)
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

In [14]:
psnr_1 = psnr1(img1, img2)
psnr_2 = psnr2(img1.unsqueeze(0), img2.unsqueeze(0))

In [16]:
psnr_1, psnr_2.item()

(4.480811357498169, 4.480811595916748)

# Result poses

In [1]:
import json
import numpy as np

### TUM
# res_file = 'results/tum/360/metadata.json'
# res_file = 'results/tum/desk/metadata.json'
# res_file = 'results/tum/desk2/metadata.json'
# res_file = 'results/tum/floor/metadata.json'
# res_file = 'results/tum/plant/metadata.json'
# res_file = 'results/tum/room/metadata.json'
# res_file = 'results/tum/rpy/metadata.json'
# res_file = 'results/tum/teddy/metadata.json'
# res_file = 'results/tum/xyz/metadata.json'

### Replica
# res_file = 'results/replica/office0/metadata.json'
# res_file = 'results/replica/office1/metadata.json'
# res_file = 'results/replica/office2/metadata.json'
# res_file = 'results/replica/office3/metadata.json'
# res_file = 'results/replica/office4/metadata.json'
# res_file = 'results/replica/room0/metadata.json'
# res_file = 'results/replica/room1/metadata.json'
# res_file = 'results/replica/room2/metadata.json'

### Waymo
# res_file = 'results/waymo/13476/metadata.json'
res_file = 'results/full/waymo/100613/metadata.json'
# res_file = 'results/waymo/106762/metadata.json'
# res_file = 'results/waymo/132384/metadata.json'
# res_file = 'results/waymo/152706/metadata.json'
# res_file = 'results/waymo/153495/metadata.json'
# res_file = 'results/waymo/158686/metadata.json'
# res_file = 'results/waymo/163453/metadata.json'
# res_file = 'results/waymo/405841/metadata.json'

# res_file = 'results_test/waymo/106762/metadata.json'


with open(res_file, 'r') as f:
    metadata = json.load(f)
print(f"metadata keys: {list(metadata.keys())}")

keyframes = metadata['keyframes']
print(f"Number of keyframes: {len(keyframes)}")

metadata keys: ['num anchors', 'num keyframes', 'num Gaussians', 'time', 'FPS', 'PSNR', 'SSIM', 'LPIPS', 'config', 'anchors', 'keyframes']
Number of keyframes: 15


In [ ]:
poses_est = []
stamps_est = []

for kf in keyframes:
    Rt = np.array(kf['Rt'])
    pose = np.linalg.inv(Rt)  ############
    poses_est.append(pose)
    
    stamp = kf['info']['frame_id']  # float
    stamps_est.append(stamp)
print(len(poses_est))

15


# GT poses TUM

In [ ]:
# gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_360/groundtruth.txt'
# gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_desk/groundtruth.txt'
# gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_desk2/groundtruth.txt'
# gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_floor/groundtruth.txt'
# gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_plant/groundtruth.txt'
# gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_room/groundtruth.txt'
# gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_rpy/groundtruth.txt'
# gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_teddy/groundtruth.txt'
gt_path = '/home/grl/datasets/tum/rgbd_dataset_freiburg1_xyz/groundtruth.txt'


gt_data = np.loadtxt(gt_path, delimiter=" ", dtype=np.str_, skiprows=3)
pose_vecs = gt_data[:, 0:].astype(np.float64)
print(f'pose_vecs.shape: {pose_vecs.shape}')

In [ ]:
import trimesh

poses_gt = []
stamps_gt = []

for k in range(len(pose_vecs)):
    stamp = pose_vecs[k][0]
    stamps_gt.append(stamp)
    
    quat = pose_vecs[k][4:]
    trans = pose_vecs[k][1:4]
    T = trimesh.transformations.quaternion_matrix(np.roll(quat, 1))
    T[:3, 3] = trans
    poses_gt.append(T)
print(len(poses_gt))

In [ ]:
# Find the closest gt pose for each estimated stamp
stamps_gt_selected = []
poses_gt_selected = []

for stamp_est in stamps_est:
    idx = np.argmin(np.abs(np.array(stamps_gt) - stamp_est))
    stamps_gt_selected.append(stamps_gt[idx])
    poses_gt_selected.append(poses_gt[idx])
    
print(len(poses_gt_selected))
print(len(poses_est))

print(len(stamps_gt_selected))
print(len(stamps_est))

# GT poses Replica

In [ ]:
import numpy as np

# gt_path = '/home/grl/datasets/replica/office0/traj.txt'
# gt_path = '/home/grl/datasets/replica/office1/traj.txt'
# gt_path = '/home/grl/datasets/replica/office2/traj.txt'
# gt_path = '/home/grl/datasets/replica/office3/traj.txt'
# gt_path = '/home/grl/datasets/replica/office4/traj.txt'
# gt_path = '/home/grl/datasets/replica/room0/traj.txt'
# gt_path = '/home/grl/datasets/replica/room1/traj.txt'
gt_path = '/home/grl/datasets/replica/room2/traj.txt'


poses_gt = []
with open(gt_path, "r") as f:
    lines = f.readlines()
for line in lines:
    pose = np.array(list(map(float, line.split()))).reshape(4, 4)
    poses_gt.append(pose)
print(f'len(poses_gt): {len(poses_gt)}')

In [ ]:
poses_gt_selected = []

for idx in stamps_est:
    idx = int(idx)
    poses_gt_selected.append(poses_gt[idx])
print(f'len(poses_gt_selected): {len(poses_gt_selected)}')

# GT poses ScanNet

In [ ]:
import numpy as np
import os

# gt_folder ='/home/grl/datasets/scannet/data/scans/scene0000_00/pose'
# gt_folder ='/home/grl/datasets/scannet/data/scans/scene0054_00/pose'
# gt_folder ='/home/grl/datasets/scannet/data/scans/scene0059_00/pose'
# gt_folder ='/home/grl/datasets/scannet/data/scans/scene0106_00/pose'
gt_folder ='/home/grl/datasets/scannet/data/scans/scene0169_00/pose'

gt_files = [
    f
    for f in os.listdir(gt_folder)
]
gt_files = sorted(gt_files, key=lambda x: int(os.path.splitext(x)[0]))

In [4]:
poses_gt = []
for gt_file in gt_files:
    pose = np.loadtxt(os.path.join(gt_folder, gt_file))
    poses_gt.append(pose)
    
print(f'len(poses_gt): {len(poses_gt)}')

len(poses_gt): 2034


In [5]:
# Find the closest gt pose for each estimated stamp
poses_gt_selected = []

for idx in stamps_est:
    idx = int(idx)
    poses_gt_selected.append(poses_gt[idx])
print(f'len(poses_gt_selected): {len(poses_gt_selected)}')

len(poses_gt_selected): 215


# GT poses Waymo

In [3]:
import numpy as np
import os

# gt_folder = '/home/grl/datasets/waymo/13476/FRONT/gt'
gt_folder = '/home/grl/datasets/waymo/100613/FRONT/gt'
# gt_folder = '/home/grl/datasets/waymo/106762/FRONT/gt'
# gt_folder = '/home/grl/datasets/waymo/132384/FRONT/gt'
# gt_folder = '/home/grl/datasets/waymo/152706/FRONT/gt'
# gt_folder = '/home/grl/datasets/waymo/153495/FRONT/gt'
# gt_folder = '/home/grl/datasets/waymo/158686/FRONT/gt'
# gt_folder = '/home/grl/datasets/waymo/163453/FRONT/gt'
# gt_folder = '/home/grl/datasets/waymo/405841/FRONT/gt'


gt_files = [f for f in os.listdir(gt_folder)]
gt_files = sorted(gt_files, key=lambda x: int(os.path.splitext(x)[0]))

In [4]:
poses_gt = []
for gt_file in gt_files:
    pose = np.loadtxt(os.path.join(gt_folder, gt_file))
    poses_gt.append(pose)
print(f'len(poses_gt): {len(poses_gt)}')

len(poses_gt): 198


In [5]:
# Find the closest gt pose for each estimated stamp
poses_gt_selected = []

for idx in stamps_est:
    idx = int(idx)
    poses_gt_selected.append(poses_gt[idx])
print(f'len(poses_gt_selected): {len(poses_gt_selected)}')

len(poses_gt_selected): 15


# GT poses KITTI

In [ ]:
import numpy as np

gt_path = '/home/grl/datasets/kitti/data_odometry_poses/dataset/poses/00.txt'


poses_gt = []
with open(gt_path, "r") as f:
    lines = f.readlines()
for line in lines:
    pose = np.array(list(map(float, line.split()))).reshape(3, 4)
    pose = np.vstack([pose, np.array([0, 0, 0, 1])])
    poses_gt.append(pose)
print(f'len(poses_gt): {len(poses_gt)}')

In [ ]:
poses_gt_selected = []

for idx in stamps_est:
    idx = int(idx)
    poses_gt_selected.append(poses_gt[idx])
print(f'len(poses_gt_selected): {len(poses_gt_selected)}')

# EVO

In [ ]:
from evo.core.trajectory import PosePath3D
from evo.core import metrics

traj_ref = PosePath3D(poses_se3=poses_gt_selected)
traj_est = PosePath3D(poses_se3=poses_est)

## Align
r_a, t_a, s = traj_est.align(traj_ref, correct_scale=True)
traj_est_aligned = traj_est

## ATE RMSE
pose_relation = metrics.PoseRelation.translation_part
data = (traj_ref, traj_est_aligned)

ape_metric = metrics.APE(pose_relation)
ape_metric.process_data(data)
ape_rmse = ape_metric.get_statistic(metrics.StatisticsType.rmse)
ape_stats = ape_metric.get_all_statistics()
print(f"RMSE ATE (m): {ape_rmse}, scale: {s}")

RMSE ATE (m): 0.768361930120193, scale: 43.95583663598047


In [ ]:
from evo.tools import plot
from matplotlib import pyplot as plt
import os

## Plot
plot_mode = plot.PlotMode.xy
# plot_mode = plot.PlotMode.xz  # kitti
# plot_mode = plot.PlotMode.xyz

fig = plt.figure()
ax = plot.prepare_axis(fig, plot_mode)
ax.set_title(f"ATE RMSE (m): {ape_rmse:.3f}, scale: {s:.3f}")
plot.traj(ax, plot_mode, traj_ref, "--", "gray", "gt")
plot.traj_colormap(
    ax,
    traj_est_aligned,
    ape_metric.error,
    plot_mode,
    min_map=ape_stats["min"],
    max_map=ape_stats["max"],
)
# ax.set_xticks([])
# ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")

ax.legend()
# plt.show()

fig_dir = os.path.dirname(res_file)
fig_path = os.path.join(fig_dir, "evo_traj.png")
fig.savefig(fig_path, dpi=300, bbox_inches='tight')